# You have more ancestors than there were people

The recursion is right. The assumption behind it is wrong, and simulating a
small population shows exactly how.

In [1]:
def ancestors(g):
    """Number of ancestor *slots* g generations back, if nobody appears twice."""
    if g == 0:
        return 1
    return 2 * ancestors(g - 1)

for g in [1, 2, 10, 20, 30]:
    print(f"{g:2} generations: {ancestors(g):>13,}")
print("world population around 1275:  ~400,000,000")

 1 generations:             2
 2 generations:             4
10 generations:         1,024
20 generations:     1,048,576
30 generations: 1,073,741,824
world population around 1275:  ~400,000,000


## A population where every person has two parents from the previous generation

Simulate 40 generations of a constant population of 500 people. For one person
in the last generation, count *distinct* ancestors per generation.

In [2]:
import random
rnd = random.Random(0)
N, G = 500, 40

# parents[g][i] = (mother index, father index) in generation g-1
parents = [None] + [[(rnd.randrange(N), rnd.randrange(N)) for _ in range(N)] for g in range(1, G + 1)]

current = {0}                      # person 0 of the last generation
for g in range(1, 16):
    gen = G - g + 1
    current = {p for i in current for p in parents[gen][i]}
    print(f"{g:2} back: {2**g:>6} slots, {len(current):>4} distinct people (population {N})")

 1 back:      2 slots,    2 distinct people (population 500)
 2 back:      4 slots,    4 distinct people (population 500)
 3 back:      8 slots,    8 distinct people (population 500)
 4 back:     16 slots,   16 distinct people (population 500)
 5 back:     32 slots,   32 distinct people (population 500)
 6 back:     64 slots,   58 distinct people (population 500)
 7 back:    128 slots,  105 distinct people (population 500)
 8 back:    256 slots,  180 distinct people (population 500)
 9 back:    512 slots,  260 distinct people (population 500)
10 back:   1024 slots,  330 distinct people (population 500)
11 back:   2048 slots,  367 distinct people (population 500)
12 back:   4096 slots,  380 distinct people (population 500)
13 back:   8192 slots,  385 distinct people (population 500)
14 back:  16384 slots,  387 distinct people (population 500)
15 back:  32768 slots,  398 distinct people (population 500)


The number of distinct ancestors stops doubling within a few generations and
saturates at the population size: the same people fill many slots. This is
**pedigree collapse**, and it is why the family "tree" is really a directed
acyclic graph.

## The same lesson in code: count each shared node once

In [3]:
def distinct_ancestors(person, gen, seen):
    """DFS over the ancestry graph with a visited set: the fix for over-counting."""
    if gen == 0 or (gen, person) in seen:
        return
    seen.add((gen, person))
    for p in parents[gen][person]:
        distinct_ancestors(p, gen - 1, seen)

seen = set()
distinct_ancestors(0, G, seen)
print("distinct ancestors over 40 generations:", len(seen), " (tree recursion would count", 2**41 - 2, ")")

distinct ancestors over 40 generations: 12461  (tree recursion would count 2199023255550 )


**What we used:** recursive definitions and what they assume about the data;
DFS with a visited set; overlapping subproblems (session 6, sections 2, 6 and 8).